In [ ]:
"""
04_results_analysis.py  [optimized]
-------------------------------------
Produces summary statistics, address-quality metrics, and bias impact plots.
Fully pandas — the input is already a processed, filtered dataset.

Usage:
    python 04_results_analysis.py <*_clean_classified.csv> [OPTIONS]

Options:
    --hotspot-file   Hotspot CSV from script 02 (auto-detected if omitted)
    --biaises-file   Path to biaises_identified.csv
    --iris-file      IRIS spatial reference (GeoJSON/shapefile)
    --revenus-file   IRIS median-income CSV
    --geocoded-dir   Directory with per-bias geocoded CSVs from script 03
    --figures-dir    Output directory for plots
    --output-dir     Output directory for CSV results
"""

In [ ]:
import argparse
import os
import sys
from datetime import datetime

In [ ]:
parser = argparse.ArgumentParser()
parser.add_argument("input_file")
parser.add_argument("--hotspot-file",  default=None)
parser.add_argument("--biaises-file",  default="./data/biaises_identified.csv")
parser.add_argument("--iris-file",     default="./data/reference/iris.geojson")
parser.add_argument("--revenus-file",  default="./data/reference/revenus_iris.csv")
parser.add_argument("--geocoded-dir",  default="./data/reference/ref_biaised_geocoded/")
parser.add_argument("--figures-dir",   default="./figures/")
parser.add_argument("--output-dir",    default="./data/outputs/")
args = parser.parse_args()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import geopandas as gpd

In [ ]:
BASE_DIR = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
sys.path.insert(0, BASE_DIR)
from methods_pandas.methods import find_pos_elem, freq_couverture, calculer_distance_euclidienne

In [ ]:
os.makedirs(args.figures_dir, exist_ok=True)
os.makedirs(args.output_dir,  exist_ok=True)

In [ ]:
date = datetime.today().strftime("%Y%m%d")
print(f"[04] start — date={date}")

In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
# §1  Load
# ═════════════════════════════════════════════════════════════════════════════
print("[04] §1 Loading data...")

In [ ]:
df = pd.read_csv(args.input_file, sep=";").drop(columns=["Unnamed: 0"], errors="ignore")
df_biaises = (
    pd.read_csv(args.biaises_file, delimiter="|",
                names=["index", "biais"], index_col="index")
    .drop(index=[None], errors="ignore")
)
biaises = pd.Series(df_biaises["biais"].to_list())
print(f"  Rows: {len(df):,}  |  Biases: {len(biaises)}")

In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
# §2  Bias position analysis
# ═════════════════════════════════════════════════════════════════════════════
print("[04] §2 Bias position analysis...")

In [ ]:
df_positions = find_pos_elem(
    df.copy(), "adr_init", biaises,
    "biais", "contains_biais", "pos_biais",
    drop_col_elem=False,
)
df_positions["comp_biais_street"] = np.nan

In [ ]:
mask_residual = (
    df_positions["contains_street_type"] &
    df_positions["contains_biais"] &
    df_positions["not_matched_brute"].notna()
)
diff = (
    df_positions.loc[mask_residual, "pos_street_type"]
    - df_positions.loc[mask_residual, "pos_biais"]
)
df_positions.loc[mask_residual, "comp_biais_street"] = np.where(
    diff > 0, "BEFORE", "AFTER"
)

In [ ]:
mask_no_street = (
    ~df_positions["contains_street_type"] &
    df_positions["contains_biais"] &
    df_positions["not_matched_brute"].notna()
)
df_positions.loc[mask_no_street, "comp_biais_street"] = "NO STREET"

In [ ]:
df_biaised = df_positions[df_positions["contains_biais"]]

In [ ]:
counts = df_biaised.groupby(["biais", "comp_biais_street"]).size().reset_index(name="n")
totals = counts.groupby("biais")["n"].sum().rename("total")
counts = counts.join(totals, on="biais")
counts["pct"] = np.round(100 * counts["n"] / counts["total"], 2)
pivot = counts.pivot(index="biais", columns="comp_biais_street", values="pct")

In [ ]:
# Frequency coverage
df_biaised = df_biaised.copy()
df_biaised["id"] = range(len(df_biaised))
df_freq = (
    freq_couverture(df_biaised, biaises.to_list(), "adr_init")
    .sort_values("cumsum", ascending=False)[["cumsum", "freq_biais_cum"]]
)
summary = pd.concat([df_freq, pivot], axis=1)

In [ ]:
out_summary = os.path.join(args.output_dir, f"bias_position_summary_{date}.csv")
summary.to_csv(out_summary, sep=";")
print(f"  Position summary → {out_summary}")
print(summary.to_string())

In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
# §3  Address-quality breakdown
# ═════════════════════════════════════════════════════════════════════════════
print("\n[04] §3 Address-quality breakdown...")

In [ ]:
# Auto-detect hotspot file
hotspot_file = args.hotspot_file
if hotspot_file is None:
    candidates = sorted(
        (f for f in os.listdir(args.output_dir) if "hotspot" in f and f.endswith(".csv")),
        key=lambda f: os.path.getmtime(os.path.join(args.output_dir, f)),
        reverse=True,
    )
    hotspot_file = os.path.join(args.output_dir, candidates[0]) if candidates else None

In [ ]:
total = len(df_positions)

In [ ]:
if hotspot_file and os.path.exists(hotspot_file):
    df_hotspot = pd.read_csv(hotspot_file, sep=";")
    df_merged  = df_positions.merge(
        df_hotspot[["adr_init", "ville_init", "cp_init", "is_hotspot"]],
        on=["adr_init", "ville_init", "cp_init"],
        how="left",
    )
    df_merged["is_hotspot"] = df_merged["is_hotspot"].fillna(False)

    clean = df_merged[
        df_merged["contains_street_type"] &
        ~df_merged["contains_biais"] &
        ~df_merged["is_hotspot"]
    ]
    print(f"  Clean addresses     : {len(clean):>7,}  ({100*len(clean)/total:.3f}%)")
    print(f"  Problematic         : {total - len(clean):>7,}  ({100*(total-len(clean))/total:.3f}%)")
else:
    print("  Hotspot file not found — skipping hotspot breakdown.")

In [ ]:
df_w  = df_positions[ df_positions["contains_street_type"]]
df_wo = df_positions[~df_positions["contains_street_type"]]
df_wb = df_positions[ df_positions["contains_biais"]]

In [ ]:
print(f"\n  With street type    : {len(df_w):>7,}  ({100*len(df_w)/total:.3f}%)")
print(f"  Without street type : {len(df_wo):>7,}  ({100*len(df_wo)/total:.3f}%)")
print(f"  With known bias     : {len(df_wb):>7,}  ({100*len(df_wb)/total:.3f}%)")

In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
# §4  Load per-bias geocoded results
# ═════════════════════════════════════════════════════════════════════════════
print("\n[04] §4 Loading geocoded bias results...")

In [ ]:
if not os.path.isdir(args.geocoded_dir):
    print("  Geocoded dir not found — skipping spatial analysis.")
    print("[04] DONE.")
    raise SystemExit(0)

In [ ]:
geo_files = [
    os.path.join(args.geocoded_dir, f)
    for f in os.listdir(args.geocoded_dir)
    if f.endswith(".csv")
]
if not geo_files:
    print("  No geocoded files found — skipping spatial analysis.")
    print("[04] DONE.")
    raise SystemExit(0)

In [ ]:
df = pd.concat(
    [pd.read_csv(f, sep=";", index_col=0) for f in geo_files],
    ignore_index=True,
)
print(f"  Loaded {len(df):,} geocoded rows across {len(geo_files)} bias files.")

In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
# §5  IRIS spatial join + income analysis
# ═════════════════════════════════════════════════════════════════════════════
print("[04] §5 IRIS spatial join and income analysis...")

In [ ]:
if not os.path.exists(args.iris_file):
    print(f"  IRIS file not found ({args.iris_file}) — skipping.")
    print("[04] DONE.")
    raise SystemExit(0)

In [ ]:
if not os.path.exists(args.revenus_file):
    print(f"  Income file not found ({args.revenus_file}) — skipping.")
    print("[04] DONE.")
    raise SystemExit(0)

In [ ]:
df_iris    = gpd.read_file(args.iris_file)
df_revenus = pd.read_csv(args.revenus_file, sep=";", dtype=str)
df_revenus["DISP_MED20"] = pd.to_numeric(df_revenus["DISP_MED20"], errors="coerce")

In [ ]:
def sjoin_iris(df_src: pd.DataFrame, lon_col: str, lat_col: str,
               df_iris: gpd.GeoDataFrame, code_col: str) -> pd.Series:
    """Spatial join → returns CODE_IRIS aligned to df_src index."""
    gdf = gpd.GeoDataFrame(
        df_src[[lon_col, lat_col]],
        geometry=gpd.points_from_xy(df_src[lon_col], df_src[lat_col]),
        crs="EPSG:4326",
    ).to_crs("EPSG:2154")
    joined = gpd.sjoin(gdf, df_iris[["CODE_IRIS", "geometry"]],
                       how="left", predicate="within")
    return joined["CODE_IRIS"].rename(code_col)

In [ ]:
df["CODE_IRIS_init"]     = sjoin_iris(df, "x_WGS84_ref", "y_WGS84_ref",
                                       df_iris, "CODE_IRIS_init")
df["CODE_IRIS_geocoded"] = sjoin_iris(df, "longitude",   "latitude",
                                       df_iris, "CODE_IRIS_geocoded")

In [ ]:
df = df.dropna(subset=["CODE_IRIS_geocoded", "CODE_IRIS_init"])

In [ ]:
# Join median incomes
df = (
    df
    .merge(df_revenus[["CODE_IRIS", "DISP_MED20"]], left_on="CODE_IRIS_geocoded",
           right_on="CODE_IRIS", how="left")
    .rename(columns={"DISP_MED20": "DISP_MED20_geocoded"})
    .drop(columns="CODE_IRIS", errors="ignore")
    .merge(df_revenus[["CODE_IRIS", "DISP_MED20"]], left_on="CODE_IRIS_init",
           right_on="CODE_IRIS", how="left")
    .rename(columns={"DISP_MED20": "DISP_MED20_init"})
    .drop(columns="CODE_IRIS", errors="ignore")
)

In [ ]:
for col in ("DISP_MED20_geocoded", "DISP_MED20_init",
            "y_WGS84_ref", "x_WGS84_ref", "latitude", "longitude"):
    df[col] = pd.to_numeric(df[col], errors="coerce")

In [ ]:
df["distance_km"] = calculer_distance_euclidienne(
    df["y_WGS84_ref"].values, df["x_WGS84_ref"].values,
    df["latitude"].values,    df["longitude"].values,
)
df["distance_m"]  = df["distance_km"] * 1000
df["diff_revenu"] = df["DISP_MED20_init"] - df["DISP_MED20_geocoded"]
df["label"]       = df["label"].str.title()

In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
# §6  Plots
# ═════════════════════════════════════════════════════════════════════════════
print("[04] §6 Generating plots...")

In [ ]:
def styled_boxplot(data: pd.DataFrame, y: str, ylabel: str,
                   title: str, ylim: tuple, out_path: str) -> None:
    fig, ax = plt.subplots(figsize=(6, 6))
    sns.boxplot(x="label", y=y, data=data,
                whis=1.5, showcaps=True, showfliers=False, ax=ax)
    sns.stripplot(x="label", y=y, data=data,
                  jitter=True, color="black", alpha=0.3, size=2, ax=ax)
    for patch in ax.patches:
        patch.set_edgecolor("red")
        patch.set_linewidth(2)
        patch.set_facecolor("none")
    ax.set(xlabel="Bias", ylabel=ylabel, title=title, ylim=ylim)
    plt.xticks(rotation=45)
    plt.tight_layout()
    fig.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"  Saved → {out_path}")

In [ ]:
styled_boxplot(
    df, "diff_revenu", "Difference of income (€)",
    "Income difference per bias",
    (0, 1000),
    os.path.join(args.figures_dir, "boxplot_diff_revenu_zoom_1000e.png"),
)

In [ ]:
styled_boxplot(
    df, "distance_km", "Distance (km)",
    "Geocoding error distance per bias",
    (0, 0.05),
    os.path.join(args.figures_dir, "boxplot_distance_km_zoom_0_50m.png"),
)

In [ ]:
# Ridge plot
g = sns.FacetGrid(df, row="label", hue="label",
                  aspect=5, height=1.2, palette="muted",
                  sharex=True, sharey=False)
g.map(sns.kdeplot, "distance_km", fill=True, alpha=0.8, linewidth=1.2)
g.map(plt.axhline, y=0, lw=1, clip_on=False)
g.set_titles(row_template="{row_name}")
g.set(yticks=[], ylabel=None)
plt.xlim(0, 1)
g.set_xlabels("Distance from reference (km)")
plt.suptitle("Ridge plot of distances per bias")
plt.tight_layout(rect=[0, 0, 1, 0.97])
out_ridge = os.path.join(args.figures_dir, "ridgeplot_distance_km.png")
g.savefig(out_ridge)
plt.close("all")
print(f"  Saved → {out_ridge}")

In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
# §7  Summary statistics table
# ═════════════════════════════════════════════════════════════════════════════
print("[04] §7 Summary statistics...")

In [ ]:
df_stats = (
    df.groupby("label")[["distance_km", "diff_revenu"]]
    .agg(["mean", "median"])
    .round(3)
)
# Flatten multi-level columns
df_stats.columns = ["dist_mean_km", "dist_med_km", "revenu_mean", "revenu_med"]
df_stats = df_stats.reset_index().rename(columns={"label": "biais"})

In [ ]:
out_stats = os.path.join(args.output_dir, f"bias_impact_stats_{date}.csv")
df_stats.to_csv(out_stats, sep=";", index=False)
print(f"  Stats table → {out_stats}")
print(df_stats.to_string(index=False))

In [ ]:
# HTML table for interactive inspection (replaces the notebook's fig.show())
out_html = os.path.join(args.output_dir, f"bias_impact_stats_{date}.html")
df_stats.to_html(out_html, index=False, border=1)
print(f"  HTML table  → {out_html}")

In [ ]:
print("[04] DONE.")